In [ ]:
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_COLAB:
    # Pin compatible versions and upgrade torchao to fix PEFT compatibility
    os.system('pip install -q transformers==4.44.0 datasets==2.19.0 peft==0.12.0 accelerate==0.33.0 torchao==0.16.0 python-dotenv')
    os.system('nvidia-smi')
    print('Packages installed. IMPORTANT: Go to Runtime → Restart session, then re-run all cells.')
else:
    print('Running locally — skipping pip install and nvidia-smi')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Option A: Upload unified_biology_dataset.json manually to /content/data/unified_biology_dataset.json
# Option B: Clone repo from GitHub
# !git clone https://github.com/<your-repo>/EduHinglish.git
import os
from dotenv import load_dotenv

# Load environment variables from .env in your Google Drive
load_dotenv('/content/drive/MyDrive/EduHinglish/.env')


In [ ]:
import json
import random
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

DATASET_PATH = Path("/content/drive/MyDrive/EduHinglish/data/unified_biology_dataset.json")
INTENT_OUTPUT_DIR = Path("/content/muril_intent_dataset")
MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
SEED = 42

INTENT_LABEL2ID = {
    "explain_concept": 0, "compare_concepts": 1, "give_example": 2,
    "formula_request": 3, "definition": 4
}
INTENT_ID2LABEL = {v: k for k, v in INTENT_LABEL2ID.items()}
INTENT_LABELS = list(INTENT_LABEL2ID.keys())

print("Loading dataset...")
with open(DATASET_PATH, encoding="utf-8") as f:
    dataset = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
queries = [e for e in dataset if e.get("is_student_query") and e.get("intent") in INTENT_LABELS]

all_examples = []
intent_counter = Counter()

for entry in queries:
    hinglish = entry.get("hinglish_roman", "")
    intent = entry["intent"]
    if not hinglish: continue
    
    tokenized = tokenizer(hinglish, max_length=MAX_LENGTH, truncation=True, padding=False)
    all_examples.append({
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": INTENT_LABEL2ID[intent]
    })
    intent_counter[intent] += 1

random.seed(SEED)
random.shuffle(all_examples)
cut = max(1, int(len(all_examples) * 0.8))
train_examples, dev_examples = all_examples[:cut], all_examples[cut:]

def _to_dataset(examples):
    return Dataset.from_dict({
        "input_ids": [e["input_ids"] for e in examples],
        "attention_mask": [e["attention_mask"] for e in examples],
        "labels": [e["labels"] for e in examples]
    })

ds = DatasetDict({"train": _to_dataset(train_examples), "dev": _to_dataset(dev_examples)})
INTENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ds.save_to_disk(str(INTENT_OUTPUT_DIR))

# Compute class weights
total = sum(intent_counter.values())
n_classes = len(INTENT_LABELS)
class_weights = {}
for intent in INTENT_LABELS:
    count = intent_counter.get(intent, 1)
    class_weights[intent] = round(total / (n_classes * count), 4)

maps = {"intent2id": INTENT_LABEL2ID, "id2intent": {str(k): v for k, v in INTENT_ID2LABEL.items()}, "class_weights": class_weights}
with open(INTENT_OUTPUT_DIR / "label_maps.json", "w") as f:
    json.dump(maps, f)
print(f"Prepared Intent data saved to {INTENT_OUTPUT_DIR}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_from_disk

ds = load_from_disk(str(INTENT_OUTPUT_DIR))
train_ds, dev_ds = ds["train"], ds["dev"]

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(INTENT_LABEL2ID), id2label=INTENT_ID2LABEL, label2id=INTENT_LABEL2ID
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.15, bias="none",
    target_modules=["query", "key", "value", "dense"]
)
model = get_peft_model(model, lora_config)

# Get weights order
weights_list = [class_weights.get(INTENT_ID2LABEL[i], 1.0) for i in range(len(INTENT_LABEL2ID))]

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device) if self.class_weights is not None else None)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="/content/muril_intent_checkpoints",
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, max_length=128)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    accuracy = (preds == labels).mean() * 100
    
    per_class, f1s = {}, []
    for class_id in range(len(INTENT_LABEL2ID)):
        intent_name = INTENT_ID2LABEL[class_id]
        tp = int(((preds == class_id) & (labels == class_id)).sum())
        fp = int(((preds == class_id) & (labels != class_id)).sum())
        fn = int(((preds != class_id) & (labels == class_id)).sum())
        
        precision = (tp / (tp + fp) * 100) if (tp + fp) > 0 else 0.0
        recall    = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        per_class[intent_name] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}
        f1s.append(f1)
        
    return {"accuracy": accuracy, "f1_macro": np.mean(f1s), "per_class": per_class}

trainer = WeightedTrainer(
    class_weights=weights_list,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
eval_result = trainer.evaluate()
print(f"Accuracy: {eval_result['eval_accuracy']:.2f}%")
print(f"F1 Macro: {eval_result['eval_f1_macro']:.2f}%")

per_class = eval_result.get("eval_per_class", {})
print("
Per-intent metrics:")
print(f"{'Intent':>18s} | {'Prec':>8s} | {'Recall':>8s} | {'F1':>8s} | {'N':>5s}")
for intent_name in INTENT_LABELS:
    m = per_class.get(intent_name, {})
    if m:
        print(f"{intent_name:>18s} | {m.get('precision', 0):>7.1f}% | {m.get('recall', 0):>7.1f}% | {m.get('f1', 0):>7.1f}% | {m.get('support', 0):>5d}")

In [ ]:
import os
merged_model = model.merge_and_unload()
# Fix non-contiguous tensors after LoRA merge
for param in merged_model.parameters():
    param.data = param.data.contiguous()
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/EduHinglish/models/muril_intent_v1"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

merged_model.save_pretrained(DRIVE_OUTPUT_DIR)
tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)
print(f"Model saved to {DRIVE_OUTPUT_DIR}")

In [ ]:
INTENT_TESTS = [
    ("Sir, nucleus ka kaam kya hota hai?", "explain_concept"),
    ("Mitochondria aur chloroplast mein kya fark hai?", "compare_concepts"),
    ("Ek example do osmosis ka?", "give_example"),
]

print("── MuRIL Intent Predictions ──")
for sent, expected in INTENT_TESTS:
    print(f'Input: "{sent}"')
    inputs = tokenizer(sent, return_tensors="pt", truncation=True).to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        conf, pred_id = torch.max(probs, dim=-1)
        pred_label = merged_model.config.id2label.get(pred_id.item(), "O")
        
    marker = "✓" if pred_label == expected else "✗"
    print(f"  Predicted: {pred_label} (conf: {conf.item():.2f})  {marker}
")